In [8]:
import glob
import pandas as pd
import numpy as np
import os
import glob
import re
import matplotlib.pyplot as plt
import random
from pathlib import Path
from tqdm import tqdm
from scipy.fft import fft
import scipy.stats as stats

In [9]:
def extract_surface_type_id(path):
    match = re.search(r'SurfaceTypeID_(\d+)', path)
    return int(match.group(1)) if match else None

data_dir = "../Datasets/Processed_Data/Labeled_Data_Without_GPS"
file_paths = glob.glob(os.path.join(data_dir, "**", "*.csv"), recursive=True)

files_df = pd.DataFrame({
    "full_path": file_paths,
    "filename": [os.path.basename(p) for p in file_paths],
})

files_df["surface_id"] = files_df["full_path"].apply(extract_surface_type_id)
files_df["device"] = files_df["filename"].apply(lambda x: x.split('_')[3])

print(files_df["surface_id"].value_counts())

files_df.head(2)

surface_id
10    406
9     403
8      98
7      96
6      90
5      66
2      57
3      40
4      30
11     28
12     20
1      10
Name: count, dtype: int64


,full_path,filename,surface_id,device
0,../Datasets/Processed_Data/Labeled_Data_Withou...,2019-02-15_SurfaceTypeID_9_SamsungGalaxyJ7_exp...,9,SamsungGalaxyJ7
1,../Datasets/Processed_Data/Labeled_Data_Withou...,2019-09-02_SurfaceTypeID_9_SamsungGalaxyS7_exp...,9,SamsungGalaxyS7


In [10]:
first_file = files_df.iloc[0]['full_path']
df_first = pd.read_csv(first_file)
df_first.head(1)

,sensorName,valueX,valueY,valueZ,timestamp
0,Gyroscope,-0.001634,-0.007524,-0.000831,1.550261e+12


# Basic feature extraction

In [11]:

# Feature extraction code goes here (not shown in the provided snippets)

def zero_crossing_rate(signal):
    return ((signal[:-1] * signal[1:]) < 0).sum()

acc_result_list = []
gyro_result_list = []

acc_window_list = []
gyro_window_list = []

WINDOW_SIZE = 1024
STEP_SIZE = WINDOW_SIZE // 2

for idx, row in tqdm(files_df.iterrows(), total=len(files_df), desc="Processing files"):
    file_path = row['full_path']
    surface_id = row['surface_id']
    
    data_df = pd.read_csv(file_path)

    # Pad if data is shorter than one window
    if len(data_df) < WINDOW_SIZE:
        pad_size = WINDOW_SIZE - len(data_df)
        pad_df   = pd.concat([data_df] * (pad_size // len(data_df) + 1)).iloc[:pad_size]
        data_df  = pd.concat([data_df, pad_df], ignore_index=True)

    # Pad last incomplete window with edge values
    remainder = len(data_df) % STEP_SIZE
    if remainder != 0:
        pad_size = WINDOW_SIZE - remainder
        pad_df   = data_df.iloc[-pad_size:].copy()
        data_df  = pd.concat([data_df, pad_df], ignore_index=True)


    for start in range(0, len(data_df) - WINDOW_SIZE + 1, STEP_SIZE):
        window = data_df[start:start + WINDOW_SIZE]

        window_xyz = {
            'window': window[['valueX', 'valueY', 'valueZ']].copy(),
            'surface_id': surface_id
        }

        svm = np.sqrt(window['valueX']**2 + window['valueY']**2 + window['valueZ']**2)

        fft_val_x = np.abs(np.fft.fft(window['valueX'].to_numpy()))[:WINDOW_SIZE//2]
        fft_val_y = np.abs(np.fft.fft(window['valueY'].to_numpy()))[:WINDOW_SIZE//2]
        fft_val_z = np.abs(np.fft.fft(window['valueZ'].to_numpy()))[:WINDOW_SIZE//2]

        features = {

            # Time domain features
            'mean_x': window['valueX'].mean(),
            'mean_y': window['valueY'].mean(),
            'mean_z': window['valueZ'].mean(),
            'std_x': window['valueX'].std(),
            'std_y': window['valueY'].std(),
            'std_z': window['valueZ'].std(),
            'var_x': window['valueX'].var(),
            'var_y': window['valueY'].var(),
            'var_z': window['valueZ'].var(),
            'sum_x': window['valueX'].sum(),
            'sum_y': window['valueY'].sum(),
            'sum_z': window['valueZ'].sum(),
            'max_x': window['valueX'].max(),
            'max_y': window['valueY'].max(),
            'max_z': window['valueZ'].max(),
            'min_x': window['valueX'].min(),
            'min_y': window['valueY'].min(),
            'min_z': window['valueZ'].min(),

            # Higher-order statistics
            'skew_x': stats.skew(window['valueX']),
            'kurt_x': stats.kurtosis(window['valueX']),
            'skew_y': stats.skew(window['valueY']),
            'kurt_y': stats.kurtosis(window['valueY']),
            'skew_z': stats.skew(window['valueZ']),
            'kurt_z': stats.kurtosis(window['valueZ']),

            # Zero Crossing Rate
            'zcr_x': zero_crossing_rate(window['valueX'].values),
            'zcr_y': zero_crossing_rate(window['valueY'].values),
            'zcr_z': zero_crossing_rate(window['valueZ'].values),

            # Single Vector Magnitude (SVM) features
            'svm_mean': svm.mean(),
            'svm_std':  svm.std(),
            'svm_max':  svm.max(),

            # Axis correlations
            'corr_xy': window['valueX'].corr(window['valueY']),
            'corr_xz': window['valueX'].corr(window['valueZ']),
            'corr_yz': window['valueY'].corr(window['valueZ']),

            # Frequency domain features
            'dominant_freq_x': np.argmax(fft_val_x),
            'spectral_energy_x': np.sum(fft_val_x**2),
            'spectral_entropy_x': -np.sum((fft_val_x/fft_val_x.sum()) * np.log(fft_val_x/fft_val_x.sum() + 1e-10)),
            
            'dominant_freq_y': np.argmax(fft_val_y),
            'spectral_energy_y': np.sum(fft_val_y**2),
            'spectral_entropy_y': -np.sum((fft_val_y/fft_val_y.sum()) * np.log(fft_val_y/fft_val_y.sum() + 1e-10)),
            
            'dominant_freq_z': np.argmax(fft_val_z),
            'spectral_energy_z': np.sum(fft_val_z**2),
            'spectral_entropy_z': -np.sum((fft_val_z/fft_val_z.sum()) * np.log(fft_val_z/fft_val_z.sum() + 1e-10)),
            
            'surface_id': surface_id,
        }

        if 'accelerometer' in file_path.lower():
            acc_result_list.append(features)
            acc_window_list.append(window_xyz)
        elif 'gyroscope' in file_path.lower():
            gyro_result_list.append(features)
            gyro_window_list.append(window_xyz)

Processing files: 100%|██████████| 1344/1344 [00:29<00:00, 45.07it/s] 


In [12]:
print(f"Extracted features from {len(acc_result_list)} accelerometer windows and {len(gyro_result_list)} gyroscope windows.")

Extracted features from 7962 accelerometer windows and 7890 gyroscope windows.


In [ ]:
# Create DataFrames from the extracted features
acc_df = pd.DataFrame(acc_result_list)
gyro_df = pd.DataFrame(gyro_result_list)

# Create output directory if it doesn't exist
output_dir = "../Datasets/ExtractedFeatures"
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Save feature CSVs
acc_df.to_csv(os.path.join(output_dir, "labeled_accelerometer_features.csv"), index=False)
gyro_df.to_csv(os.path.join(output_dir, "labeled_gyroscope_features.csv"), index=False)

# Save accelerometer windows as pickle — preserves list-of-DataFrames structure
import pickle

acc_windows_path = os.path.join(output_dir, "labeled_accelerometer_windows.pkl")
with open(acc_windows_path, "wb") as f:
    pickle.dump(acc_window_list, f)

print(f"Accelerometer features saved: {len(acc_df)} samples")
print(f"Gyroscope features saved: {len(gyro_df)} samples")
print(f"Accelerometer windows saved: {len(acc_window_list)} windows → {acc_windows_path}")

Accelerometer features saved: 7962 samples
Gyroscope features saved: 7890 samples
Accelerometer windows saved: 7962 windows → ../Datasets/ExtractedFeatures/labeled_accelerometer_windows.pkl
